WEEK 1

Extracting text from pdf

In [ ]:
from pdfminer.high_level import extract_text

pdf_path = "C:/Users/HPVictus/ai_resume_project/resumes/resume.pdf"

text = extract_text(pdf_path)
print(text)


Extracting text from doc

In [ ]:
import docx2txt

docx_path = "C:/Users/HPVictus/ai_resume_project/resumes/resume doc.docx"

text = docx2txt.process(docx_path)
print(text)


Basic text cleaning

In [ ]:
import re

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\n+', ' ', text)        # remove new lines
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)  # remove symbols
    text = re.sub(r'\s+', ' ', text)        # extra spaces
    return text.strip()


In [ ]:
cleaned_text = clean_text(text)
print(cleaned_text)


PDF & DOCX -> single reusable pipeline + testing

In [ ]:
from pdfminer.high_level import extract_text
import docx2txt
import os
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_resume_text(file_path):
    if file_path.endswith(".pdf"):
        text = extract_text(file_path)
    elif file_path.endswith(".docx"):
        text = docx2txt.process(file_path)
    else:
        return None
    
    return clean_text(text)


In [ ]:
resume_files = [
    "C:/Users/HPVictus/ai_resume_project/resumes/resume.pdf",
    "C:/Users/HPVictus/ai_resume_project/resumes/resume doc.docx"
]

for file in resume_files:
    print(f"\n--- {file} ---")
    print(extract_resume_text(file)[:500])  # first 500 chars


WEEK 2

Job Description processing

step 1 : sample JD

In [ ]:
job_description = """
We are looking for a Data Analyst with strong skills in Python, SQL,
machine learning, data visualization, and statistics.
Experience with Pandas, NumPy, and Power BI is a plus.
"""


step 2: reuse cleaning function

In [ ]:
clean_jd = clean_text(job_description)
print(clean_jd)


Tokenization & Stopword removal(spaCy)

step 1: load spacy model

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")


step 2 : tokenization + stopword removal function

In [ ]:
def tokenize_and_remove_stopwords(text):
    doc = nlp(text)
    tokens = [token.text for token in doc 
              if not token.is_stop and not token.is_space]
    return tokens


step 3: apply to resume & JD

In [ ]:
resume_tokens = tokenize_and_remove_stopwords(cleaned_text)
jd_tokens = tokenize_and_remove_stopwords(clean_jd)

print(resume_tokens[:30])
print(jd_tokens[:30])


Lemmatization

step 1: Lemmetization function

In [ ]:
def lemmatize_tokens(text):
    doc = nlp(text)
    lemmas = [
        token.lemma_ for token in doc
        if not token.is_stop and not token.is_space
    ]
    return lemmas


step 2: apply to resume & JD

In [ ]:
resume_lemmas = lemmatize_tokens(cleaned_text)
jd_lemmas = lemmatize_tokens(clean_jd)

print(resume_lemmas[:30])
print(jd_lemmas[:30])


Final NLP preprocessing pipeline

step 1: final preprocessing funcion

In [ ]:
def preprocess_text(text):
    text = clean_text(text)
    doc = nlp(text)
    lemmas = [
        token.lemma_
        for token in doc
        if not token.is_stop and not token.is_space
    ]
    return " ".join(lemmas)


step 2 : apply to resume & JD

In [ ]:
final_resume_text = preprocess_text(cleaned_text)
final_jd_text = preprocess_text(clean_jd)

print(final_resume_text[:300])
print(final_jd_text[:300])


Testing , Validation & wrap-up

step 1: test with multiple resumes

In [ ]:
resume_files = [
    "C:/Users/HPVictus/ai_resume_project/resumes/resume.pdf",
    "C:/Users/HPVictus/ai_resume_project/resumes/resume doc.docx"
]

processed_resumes = []

for file in resume_files:
    raw_text = extract_resume_text(file)
    processed_resumes.append(preprocess_text(raw_text))

for r in processed_resumes:
    print(r[:200])


step 2: validate JD processing

In [ ]:
print(final_jd_text)


WEEK 3 - RESUME -JD MATCHING & SCORING PIPELINE

TF-IDF vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer


In [ ]:
documents = processed_resumes + [final_jd_text]


In [ ]:
vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(documents)


In [ ]:
tfidf_matrix.shape


Cosine Similarity Calculation

step 1: import cosine similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


step 2: seperate resume & JD vectors

In [ ]:
resume_vectors = tfidf_matrix[:-1]   # all resumes
jd_vector = tfidf_matrix[-1]         # job description


step 3: calculate similarity

In [ ]:
similarity_scores = cosine_similarity(resume_vectors, jd_vector)


step 4: view output

In [ ]:
similarity_scores


Match Percentage Calculation

step 1: convert to percentage

In [ ]:
match_percentages = similarity_scores.flatten() * 100
match_percentages


step 2: attach resume names

In [ ]:
resume_names = ["resume1.pdf", "resume2.docx"]

for name, score in zip(resume_names, match_percentages):
    print(f"{name} : {score:.2f}% match")


Ranking & Shortlisting

step 1: create ranking data

In [ ]:
results = list(zip(resume_names, match_percentages))


step 2: sort by score(desc)

In [ ]:
ranked_results = sorted(results, key=lambda x: x[1], reverse=True)
ranked_results


step 3: display ranking

In [ ]:
for rank, (name, score) in enumerate(ranked_results, start=1):
    print(f"Rank {rank}: {name} - {score:.2f}%")


optional shortlisting rule

In [ ]:
threshold = 60

shortlisted = [r for r in ranked_results if r[1] >= threshold]
shortlisted


ADVANCED 

skill dictionary & extraction

step 1: skill dictionary create

In [ ]:
SKILLS = [
    "python", "sql", "machine learning", "deep learning",
    "data analysis", "nlp", "pandas", "numpy",
    "power bi", "tableau", "excel",
    "flask", "django", "fastapi",
    "scikit learn", "tensorflow", "pytorch"
]


step 2: skill extraction function

In [ ]:
def extract_skills(text, skill_list):
    extracted = []
    for skill in skill_list:
        if skill in text:
            extracted.append(skill)
    return list(set(extracted))


step 3: apply to resume & JD

In [ ]:
resume_skills = extract_skills(final_resume_text, SKILLS)
jd_skills = extract_skills(final_jd_text, SKILLS)

print("Resume Skills:", resume_skills)
print("JD Skills:", jd_skills)


SKILL BASED MATCH SCORE

step 1: skill match function

In [ ]:
def skill_match_score(resume_skills, jd_skills):
    matched = list(set(resume_skills) & set(jd_skills))
    missing = list(set(jd_skills) - set(resume_skills))
    
    if len(jd_skills) == 0:
        score = 0
    else:
        score = (len(matched) / len(jd_skills)) * 100
    
    return score, matched, missing


step 2 : apply function

In [ ]:
skill_score, matched_skills, missing_skills = skill_match_score(
    resume_skills, jd_skills
)

print("Skill Match % :", skill_score)
print("Matched Skills :", matched_skills)
print("Missing Skills :", missing_skills)


EXPLAINABLE AI 

step 1: explainable ai output function

In [ ]:
def explain_result(skill_score, matched, missing):
    explanation = {
        "Skill Match %": round(skill_score, 2),
        "Matched Skills": matched,
        "Missing Skills": missing,
        "Recommendation": ""
    }
    
    if skill_score == 100:
        explanation["Recommendation"] = "Excellent fit for the role"
    elif skill_score >= 70:
        explanation["Recommendation"] = "Good fit, improve missing skills"
    elif skill_score >= 40:
        explanation["Recommendation"] = "Partial fit, skill improvement needed"
    else:
        explanation["Recommendation"] = "Not a good fit, significant upskilling required"
    
    return explanation


step 2: generate explainable output

In [ ]:
explanation = explain_result(skill_score, matched_skills, missing_skills)
explanation


HYBRID SCORE(SKILL+TF-IDF)

step 1: normalize TF-IDF SCORE

In [ ]:
tfidf_score = match_percentages[0]  # resume1 example (already in %)


step 2: hybrid score formula

In [ ]:
def hybrid_score(skill_score, tfidf_score, skill_weight=0.7, tfidf_weight=0.3):
    return (skill_score * skill_weight) + (tfidf_score * tfidf_weight)


step 3: calculate final score

In [ ]:
final_score = hybrid_score(skill_score, tfidf_score)

print("Skill Score  :", skill_score)
print("TF-IDF Score :", tfidf_score)
print("Final Hybrid Score :", round(final_score, 2))
